In [ ]:
import numpy as np
import pandas as pd
import os
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report
from dotenv import load_dotenv
import joblib


In [14]:
load_dotenv()

y_test_path = os.getenv("Y_TEST_TSV_PATH")
X_test_feature_selected_path = os.getenv("X_TEST_FEATURE_SELECTED_CSV_PATH")
output_dir = os.getenv("OUTPUT_DIR")
random_seed = int(os.getenv("RANDOM_SEED"))
n_jobs = int(os.getenv("N_JOBS", -1))
rf_model_path = os.getenv("RF_MODEL_PATH")
selected_features_path = os.getenv("SELECTED_FEATURES_PATH")
pipeline_path = os.getenv("PIPELINE_PATH")
hyperparams_path = os.getenv("HYPERPARAMS_PATH")

# Print configuration
print(f"y_test_path: {y_test_path}")
print(f"X_test_feature_selected_path: {X_test_feature_selected_path}")
print(f"output_dir: {output_dir}")
print(f"random_seed: {random_seed}")
print(f"n_jobs: {n_jobs}")
print(f"rf_model_path: {rf_model_path}")
print(f"selected_features_path: {selected_features_path}")
print(f"pipeline_path: {pipeline_path}")
print(f"hyperparams_path: {hyperparams_path}")

y_test_path: /home/mariya/Data/pallas/processed/dnam/dnam_test_y.tsv
X_test_feature_selected_path: /home/mariya/Data/pallas/processed/dnam/dnam_test_X_feature_selected.csv
output_dir: /home/mariya/Data/pallas/training/dnam
random_seed: 42
n_jobs: 10
rf_model_path: /home/mariya/Data/pallas/training/dnam/dnam_final_model.pkl
selected_features_path: /home/mariya/Data/pallas/training/dnam/dnam_selected_features.pkl
pipeline_path: /home/mariya/Data/pallas/training/dnam/dnam_pipeline.pkl
hyperparams_path: /home/mariya/Data/pallas/training/dnam/dnam_hyperparams.PKL


In [15]:
X_test = pd.read_csv(X_test_feature_selected_path, index_col=0)
y_test = pd.read_csv(y_test_path, sep="\t", index_col=0)
X_test, y_test

(            cg07124876  cg07067189  cg01739725  cg11237495  cg10277811  \
 id                                                                       
 ALL_1154      0.851290    0.942538    0.950721    0.878240    0.910321   
 GSM5734757    0.805903    0.899436    0.097319    0.731847    0.678789   
 ALL_657       0.911846    0.932911    0.653037    0.885070    0.751230   
 GSM5734711    0.768504    0.083034    0.400290    0.713807    0.775872   
 ALL_1252      0.851273    0.922816    0.616707    0.875467    0.836878   
 ...                ...         ...         ...         ...         ...   
 ALL_1062      0.872726    0.681418    0.943831    0.874825    0.906218   
 GSM5735128    0.653670    0.114238    0.840407    0.780117    0.652049   
 ALL_944       0.915517    0.959914    0.941825    0.902001    0.883059   
 ALL_693       0.921288    0.956779    0.871876    0.911855    0.927344   
 ALL_422       0.930313    0.952985    0.751021    0.893104    0.875638   
 
             cg27471989

In [16]:
# align the indices
X_test = X_test.loc[y_test.index]
X_test

,cg07124876,cg07067189,cg01739725,cg11237495,cg10277811,cg27471989,cg22275872,cg03509771,cg08527226,cg07145988,...,cg05088898,cg09499856,cg01937808,cg12430029,cg20981451,cg10004780,cg11869770,cg19922137,cg01216388,cg17799946
id,,,,,,,,,,,,,,,,,,,,,
ALL_1154,0.851290,0.942538,0.950721,0.878240,0.910321,0.904852,0.896068,0.930047,0.824901,0.761829,...,0.106109,0.701635,0.936158,0.409506,0.812953,0.941896,0.596240,0.869049,0.278321,0.565912
GSM5734757,0.805903,0.899436,0.097319,0.731847,0.678789,0.398433,0.834762,0.896084,0.559936,0.847912,...,0.190383,0.013201,0.853394,0.788562,0.146752,0.076829,0.028647,0.011813,0.540085,0.022754
ALL_657,0.911846,0.932911,0.653037,0.885070,0.751230,0.883731,0.913067,0.909496,0.898097,0.769804,...,0.085862,0.537000,0.905804,0.479425,0.531536,0.674084,0.632141,0.648284,0.638407,0.618994
GSM5734711,0.768504,0.083034,0.400290,0.713807,0.775872,0.679314,0.128658,0.060866,0.298750,0.837054,...,0.130317,0.023572,0.925952,0.907965,0.303439,0.027711,0.221230,0.189593,0.462296,0.020728
ALL_1252,0.851273,0.922816,0.616707,0.875467,0.836878,0.860873,0.791443,0.916828,0.657288,0.677541,...,0.123059,0.541554,0.906431,0.174842,0.573884,0.732816,0.650132,0.697559,0.636121,0.529523
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
ALL_1062,0.872726,0.681418,0.943831,0.874825,0.906218,0.904018,0.895146,0.910064,0.852886,0.952502,...,0.222639,0.707570,0.915788,0.638989,0.871357,0.951734,0.835975,0.891712,0.751155,0.840896
GSM5735128,0.653670,0.114238,0.840407,0.780117,0.652049,0.784059,0.658894,0.907624,0.531436,0.701468,...,0.148279,0.016416,0.854199,0.557266,0.099116,0.064059,0.078856,0.183516,0.558348,0.046381
ALL_944,0.915517,0.959914,0.941825,0.902001,0.883059,0.845273,0.912530,0.938887,0.869743,0.273895,...,0.096714,0.728846,0.945059,0.135900,0.882566,0.503166,0.611418,0.846180,0.739093,0.430099


In [17]:
y_test = y_test.iloc[:, 0].values # Convert to 1D numpy array
y_test

array(['B-ALL, PAX5alt', 'AML, KMT2A-r', 'B-ALL, high hyperdiploid',
       'AML, CBFB::MYH11', 'B-ALL, high hyperdiploid', 'Non-leukemic',
       'AML, KMT2A-r', 'AML, RUNX1::RUNX1T1', 'B-ALL, high hyperdiploid',
       'B-ALL, high hyperdiploid', 'B-ALL, ETV6::RUNX1',
       'B-ALL, high hyperdiploid', 'AML, RUNX1::RUNX1T1',
       'AML, RUNX1::RUNX1T1', 'B-ALL, high hyperdiploid',
       'AML, RUNX1::RUNX1T1', 'B-ALL, ETV6::RUNX1', 'B-ALL, KMT2A-r',
       'Non-leukemic', 'AML, CBFB::MYH11', 'B-ALL, high hyperdiploid',
       'AML, RUNX1::RUNX1T1', 'AML, KMT2A-r', 'AML, RUNX1::RUNX1T1',
       'B-ALL, high hyperdiploid', 'B-ALL, ETV6::RUNX1',
       'B-ALL, ETV6::RUNX1', 'B-ALL, ETV6::RUNX1', 'AML, CBFB::MYH11',
       'AML, KMT2A-r', 'B-ALL, ETV6::RUNX1', 'B-ALL, high hyperdiploid',
       'B-ALL, ETV6::RUNX1', 'AML, KMT2A-r', 'T-ALL', 'AML, KMT2A-r',
       'B-ALL, high hyperdiploid', 'AML, KMT2A-r', 'AML, KMT2A-r',
       'T-ALL', 'B-ALL, ETV6::RUNX1', 'B-ALL, PAX5alt',
       'B

In [ ]:
# Load model
final_rf = joblib.load(rf_model_path)

# Load selected features
selected_features = joblib.load(selected_features_path)

# Use selected features for test set
y_pred_final = final_rf.predict(X_test)

# Print size of selected features
print(f"Number of selected features: {len(selected_features)}")

# Print shape of X_test_selected
print(f"Shape of X_test_selected: {X_test.shape}")



/home/mariya/miniconda3/envs/pallas/lib/python3.10/site-packages/sklearn/base.py:442: InconsistentVersionWarning: Trying to unpickle estimator DecisionTreeClassifier from version 1.4.2 when using version 1.7.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/home/mariya/miniconda3/envs/pallas/lib/python3.10/site-packages/sklearn/base.py:442: InconsistentVersionWarning: Trying to unpickle estimator RandomForestClassifier from version 1.4.2 when using version 1.7.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


Number of selected features: 33836
Shape of X_test_selected: (311, 33836)

Test set classification report (final optimized model):
                          precision    recall  f1-score   support

      AML, APL-PML::RARA      1.000     1.000     1.000         1
     AML, CBFA2T3::GLIS2      0.833     0.833     0.833         6
        AML, CBFB::MYH11      1.000     0.867     0.929        15
           AML, CEBPAalt      1.000     0.500     0.667         2
        AML, DEK::NUP214      1.000     1.000     1.000         4
             AML, ETV6-r      1.000     0.500     0.667         4
            AML, KAT6A-r      0.000     0.000     0.000         1
            AML, KMT2A-r      0.723     0.971     0.829        35
            AML, MECOM-r      0.000     0.000     0.000         1
            AML, NPM1alt      0.000     0.000     0.000         2
            AML, NUP98-r      0.625     0.500     0.556        10
     AML, PICALM::MLLT10      1.000     1.000     1.000         3
        AM

/home/mariya/miniconda3/envs/pallas/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/mariya/miniconda3/envs/pallas/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/mariya/miniconda3/envs/pallas/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metri

In [19]:
print("\nTest set classification report (final optimized model):")
print(classification_report(y_test, y_pred_final, digits=3))



Test set classification report (final optimized model):
                          precision    recall  f1-score   support

      AML, APL-PML::RARA      1.000     1.000     1.000         1
     AML, CBFA2T3::GLIS2      0.833     0.833     0.833         6
        AML, CBFB::MYH11      1.000     0.867     0.929        15
           AML, CEBPAalt      1.000     0.500     0.667         2
        AML, DEK::NUP214      1.000     1.000     1.000         4
             AML, ETV6-r      1.000     0.500     0.667         4
            AML, KAT6A-r      0.000     0.000     0.000         1
            AML, KMT2A-r      0.723     0.971     0.829        35
            AML, MECOM-r      0.000     0.000     0.000         1
            AML, NPM1alt      0.000     0.000     0.000         2
            AML, NUP98-r      0.625     0.500     0.556        10
     AML, PICALM::MLLT10      1.000     1.000     1.000         3
        AML, RBM15::MKL1      0.000     0.000     0.000         2
     AML, RUNX1::R

/home/mariya/miniconda3/envs/pallas/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/mariya/miniconda3/envs/pallas/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/mariya/miniconda3/envs/pallas/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metri

In [20]:
# ----------------------
# Extract top cpg sites from final model
# ----------------------
# Get feature importances from final RF
importances = final_rf.feature_importances_

# Rank genes by importance
gene_importances = pd.DataFrame({
    "Cpg Site": selected_features,
    "Importance": importances
}).sort_values(by="Importance", ascending=False)

print("\nTop 20 cpg sites driving classification (final model):")
print(gene_importances.head(20))


Top 20 cpg sites driving classification (final model):
         Cpg Site  Importance
5188   cg06891858    0.011384
10363  cg15172966    0.008927
12836  cg03380518    0.008366
3322   cg21359950    0.007169
5748   cg24851084    0.006647
25420  cg15168000    0.006026
19449  cg00393348    0.005986
2326   cg02835823    0.005274
14811  cg12871285    0.005071
31194  cg11762703    0.004860
23535  cg16391792    0.004650
12156  cg02064106    0.004363
30591  cg11986756    0.004282
33711  cg16288865    0.004202
10663  cg03684571    0.004055
30281  cg13897947    0.003926
9740   cg05955831    0.003874
27253  cg10253457    0.003856
28477  cg04961270    0.003803
31798  cg10861599    0.003723


In [21]:
# How many of the feature importances are zero?
num_zero_importances = np.sum(importances == 0)
print(f"\nNumber of features with zero importance: {num_zero_importances} out of {len(importances)}")

# Print lowest importance genes as well
print("\nBottom 20 genes by importance (final model):")
print(gene_importances.tail(20))


Number of features with zero importance: 20511 out of 33836

Bottom 20 genes by importance (final model):
         Cpg Site  Importance
20098  cg19921577         0.0
20097  cg01376079         0.0
20096  cg07139527         0.0
20095  cg25746394         0.0
20123  cg23950767         0.0
20067  cg26308280         0.0
20065  cg11078090         0.0
20064  cg08118747         0.0
20082  cg17674725         0.0
20060  cg02524336         0.0
20091  cg18139928         0.0
20090  cg00961326         0.0
20089  cg16846532         0.0
20088  cg13250285         0.0
20087  cg14794027         0.0
20086  cg15417287         0.0
20085  cg16294414         0.0
20084  cg10923313         0.0
20083  cg06955979         0.0
20063  cg26155520         0.0
